# Séance 2 — P1 : 2.1 Le découpage · 2.5 Le premier modèle
**Sujet 05 · Santander Satisfaction** — cible `TARGET` (≈ 3,96 % de mécontents), métrique **AUC**, plancher métier **0,6890** (README).

Ce carnet est autonome : il charge lui-même `train.csv`. Exécuter les cellules dans l'ordre.

## 0. Réglages et chargement

In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd

SEED = 42            # graine commune à toute l'équipe
PLANCHER = 0.6890    # règle métier (README)
CIBLE = "TARGET"

def trouver_data():
    ici = Path.cwd().resolve()
    for c in [ici / "data", ici.parent / "data", ici.parent.parent / "data"]:
        if (c / "train.csv").is_file():
            return c
    raise FileNotFoundError(f"train.csv introuvable depuis {ici}")

DATA = trouver_data()
train = pd.read_csv(DATA / "train.csv")
features = [c for c in train.columns if c not in ("ID", CIBLE)]
X, y = train[features], train[CIBLE]
print("dossier :", DATA)
print("train :", train.shape, "| mécontents :", round(y.mean() * 100, 2), "%")

dossier : /home/vino-yd/santander-satisfaction/santander-satisfaction/data
train : (60816, 371) | mécontents : 3.96 %


## 2.1 Le découpage
Choix à justifier : **stratifié** (cible rare), **80 / 20**, **graine fixe**, fait **une seule fois avant tout prétraitement**, identique pour toute l'équipe.

In [2]:
from sklearn.model_selection import train_test_split

idx_app, idx_val = train_test_split(
    train.index, test_size=0.20, stratify=y, random_state=SEED
)
X_app, y_app = X.loc[idx_app], y.loc[idx_app]
X_val, y_val = X.loc[idx_val], y.loc[idx_val]

print("apprentissage :", X_app.shape, "| part de 1 :", round(y_app.mean() * 100, 2), "%")
print("validation    :", X_val.shape, "| part de 1 :", round(y_val.mean() * 100, 2), "%")
print("ID en commun  :", len(set(train.loc[idx_app, "ID"]) & set(train.loc[idx_val, "ID"])))
print("doublons dans train (hors ID/TARGET) :", int(train.duplicated(subset=features).sum()))

apprentissage : (48652, 369) | part de 1 : 3.96 %
validation    : (12164, 369) | part de 1 : 3.95 %
ID en commun  : 0
doublons dans train (hors ID/TARGET) : 3788


**À noter**
- tailles : 60 816 clients au total, découpés en 48 652 pour l'apprentissage (80 %) et 12 164 pour la validation (20 %). Chacun a 369 variables.
- part de mécontents : 3,96 % en apprentissage et 3,95 % en validation, donc quasi identique grâce à la stratification
- ID communs aux deux groupes : 0
- doublons dans train : 3 788, à signaler à P4 (tâche 1.7). Leur effet sur la validation n'est pas encore mesuré.
- découpage dans le temps : à vérifier dans docs/enonce.md

## 2.5 Le premier modèle
Une **référence**, pas un modèle optimisé. Tout est appris sur l'apprentissage uniquement.

In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# colonnes constantes repérées sur l'apprentissage seulement
cols = [c for c in X_app.columns if X_app[c].nunique(dropna=False) > 1]
print("colonnes gardées :", len(cols), "sur", X_app.shape[1])

modeles = {
    "régression logistique": make_pipeline(
        SimpleImputer(strategy="median"), StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced")),
    "gradient boosting": HistGradientBoostingClassifier(random_state=SEED),
}

resultats = []
for nom, m in modeles.items():
    t = time.time()
    m.fit(X_app[cols], y_app)
    auc_app = roc_auc_score(y_app, m.predict_proba(X_app[cols])[:, 1])
    auc_val = roc_auc_score(y_val, m.predict_proba(X_val[cols])[:, 1])
    resultats.append((nom, round(auc_app, 4), round(auc_val, 4), round(time.time() - t, 1)))

res = pd.DataFrame(resultats, columns=["modèle", "AUC apprentissage", "AUC validation", "secondes"])
res["écart au plancher"] = (res["AUC validation"] - PLANCHER).round(4)
res["sur-apprentissage"] = (res["AUC apprentissage"] - res["AUC validation"]).round(4)
res

colonnes gardées : 320 sur 369


,modèle,AUC apprentissage,AUC validation,secondes,écart au plancher,sur-apprentissage
0,régression logistique,0.8216,0.7756,24.4,0.0866,0.0460
1,gradient boosting,0.9034,0.8319,5.6,0.1429,0.0715


**Lecture**
- colonnes utilisées : 320 sur 369 (49 colonnes constantes retirées)
- AUC validation : 0,7756 pour la régression logistique et 0,8319 pour le gradient boosting. Le plancher du README est 0,6890.
- écart au plancher : la régression logistique le dépasse de 0,0866 et le gradient boosting de 0,1429
- sur-apprentissage (AUC apprentissage moins AUC validation) : 0,0460 pour la régression logistique et 0,0715 pour le gradient boosting
- temps d'exécution : 24,4 secondes et 5,6 secondes, très en dessous de la limite de 10 minutes
- ligne pour le tableau d'essai (2.7) : gradient boosting, AUC validation 0,8319, bat le plancher de 0,1429. Ce score peut être un peu gonflé par les doublons, à confirmer.